In [24]:
# ============= OK OKOK ========================
# NODE 1: BASIC — Chỉ 1 endpoint, không package
# ============================================

import requests
import json
import time
import re


def get_suggestions(query: str) -> list:
    """
    Chỉ dùng 1 endpoint Google Suggest public.
    Trả về list string suggestions.
    """
    url = "https://suggestqueries.google.com/complete/search"
    
    params = {
        "client": "firefox",  # Firefox = JSON thuần, không JSONP
        "hl": "vi",
        "q": query
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/109.0",
        "Accept": "application/json, text/javascript, */*",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": "https://www.google.com/"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=15)
        print(f"  Status: {response.status_code}")
        print(f"  URL: {response.url[:80]}...")
        
        if response.status_code != 200:
            print(f"  Response: {response.text[:200]}")
            return []
        
        # Firefox client trả về JSON: [query, [suggestions]]
        data = response.json()
        print(f"  Raw response type: {type(data)}")
        print(f"  Raw: {str(data)[:200]}")
        
        if isinstance(data, list) and len(data) >= 2:
            suggestions = data[1]
            if isinstance(suggestions, list):
                # Lọc bỏ query gốc và quá ngắn
                return [s for s in suggestions if s != query and len(s) > 3]
        
        return []
        
    except Exception as e:
        print(f"  ERROR: {e}")
        return []


def expand_query(query: str) -> list:
    """
    Mở rộng query để lấy nhiều suggestions.
    """
    all_results = set()
    
    # 1. Query gốc
    print(f"\n[1] Base: '{query}'")
    base = get_suggestions(query)
    all_results.update(base)
    print(f"    → {len(base)} suggestions")
    time.sleep(0.5)
    
    # 2. Thêm " " + a-z (mỗi chữ cái cho 1 suggestion mới)
    print(f"\n[2] Alphabet expansion...")
    for char in "abcdefghijklmnopqrstuvwxyz":
        new_query = f"{query} {char}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    # 3. Thêm số (giá, số người, số ngày)
    print(f"\n[3] Number expansion...")
    for num in "0123456789":
        new_query = f"{query} {num}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    # 4. Thêm từ đệm tiết lộ intent
    print(f"\n[4] Intent words...")
    intent_words = ["giá", "gần", "ở", "cho", "review", "nào", "ngon", "rẻ", "đẹp"]
    for word in intent_words:
        new_query = f"{query} {word}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    return list(all_results)


def simple_intent_tag(queries: list) -> dict:
    """
    Gắn tag đơn giản — không AI.
    """
    tags = {
        "budget": ["giá", "rẻ", "tiết kiệm", "bao nhiêu", "chi phí"],
        "location": ["gần", "ở", "tại", "đường", "khu"],
        "food": ["hải sản", "món", "ăn", "nhậu", "buffet", "nướng"],
        "experience": ["review", "trải nghiệm", "ngon", "đẹp", "view"],
        "family": ["gia đình", "nhóm", "đoàn", "nhiều người"],
        "question": ["?", "là gì", "ở đâu", "có nên", "nào"]
    }
    
    result = {tag: [] for tag in tags}
    result["other"] = []
    
    for q in queries:
        q_lower = q.lower()
        matched = False
        for tag, keywords in tags.items():
            if any(kw in q_lower for kw in keywords):
                result[tag].append(q)
                matched = True
                break
        if not matched:
            result["other"].append(q)
    
    return result


# ============================================
# RUN
# ============================================

if __name__ == "__main__":
    QUERY = "nhà hàng quán nhậu hải sản Đà Nẵng"
    
    print("=" * 60)
    print("NODE 1: BASIC SUGGEST COLLECTOR")
    print("=" * 60)
    
    # Thu thập
    all_suggestions = expand_query(QUERY)
    
    print(f"\n{'='*60}")
    print(f"TOTAL: {len(all_suggestions)} unique suggestions")
    print(f"{'='*60}")
    
    # Phân loại
    tagged = simple_intent_tag(all_suggestions)
    
    print(f"\n--- TAGGED ---")
    for tag, items in sorted(tagged.items(), key=lambda x: -len(x[1])):
        if items:
            print(f"\n[{tag.upper()}] ({len(items)} items)")
            for item in items[:5]:
                print(f"  • {item}")
    
    # Lưu
    output = {
        "seed": QUERY,
        "total": len(all_suggestions),
        "suggestions": all_suggestions,
        "tagged": tagged
    }
    
    with open("node1_basic.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Saved: node1_basic.json")

NODE 1: BASIC SUGGEST COLLECTOR

[1] Base: 'nhà hàng quán nhậu hải sản Đà Nẵng'
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng', []]
    → 0 suggestions

[2] Alphabet expansion...
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng a', ['nhà hàng quán nhậu hải sản đà nẵng a lưới', 'nhà hàng quán nhậu hải sản đà nẵng a-z'], [], {'google:suggestdetail': [{'mp': '… ', 't': 'a lưới'}, {'mp': '… '
    'nhà hàng quán nhậu hải sản Đà ' → 2
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng b', ['nhà hàng quán nhậu hải sản đà nẵng bình dân', 'nhà hàng quán nhậu hải sản đà nẵng bao nhiêu tiền'], [

In [25]:
# ============= OK OKOK ========================
import asyncio
from urllib.parse import quote
from bs4 import BeautifulSoup
import nodriver as uc

QUERY = "nhà hàng quán nhậu hải sản Đà Nẵng"


# =========================
# CLEAN
# =========================
def clean(text: str):
    if not text:
        return None

    text = " ".join(text.split())

    BAD = [
        "facebook",
        "tripadvisor",
        "đăng nhập",
        "chính sách",
        "hỗ trợ",
        "video",
        "youtube",
        "maps",
        "translate",
        "google",
        "javascript",
        "cookie"
    ]

    if (
        len(text) < 5
        or len(text) > 160
        or any(x in text.lower() for x in BAD)
    ):
        return None

    return text


def is_valid_url(url: str):
    if not url or not url.startswith("http"):
        return False

    BAD_URL = [
        "google.com/search",
        "googleusercontent",
        "webcache",
        "/settings",
        "accounts.google"
    ]

    return not any(x in url for x in BAD_URL)


# =========================
# NODE 1 ENGINE
# =========================
async def node1_serp():

    browser = await uc.start(headless=False)

    try:
        page = await browser.get(
            f"https://www.google.com/search?q={quote(QUERY)}&hl=vi"
        )

        await asyncio.sleep(3)

        soup = BeautifulSoup(await page.get_content(), "html.parser")

        # =========================
        # 1. TOP URLS (Organic)
        # =========================
        top_urls = []
        seen = set()

        for a in soup.select("a"):
            h3 = a.select_one("h3")
            if not h3:
                continue

            url = a.get("href")
            title = clean(h3.get_text(" ", strip=True))

            if not title or not is_valid_url(url):
                continue

            key = url.split("#")[0]
            if key in seen:
                continue

            seen.add(key)

            domain = url.split("/")[2] if "://" in url else ""

            top_urls.append({
                "position": len(top_urls) + 1,
                "title": title,
                "url": url,
                "domain": domain,
                "type": "organic"
            })

        # =========================
        # 2. PEOPLE ALSO ASK (PAA)
        # =========================
        paa = []

        for el in soup.select('div[jsname="N760b"]'):
            q = clean(el.get_text(" ", strip=True))
            if q and "?" in q:
                paa.append({
                    "question": q,
                    "intent": "expansion"
                })

        paa = paa[:5]

        # =========================
        # 3. RELATED SEARCHES
        # =========================
        related = []

        for a in soup.select("a"):
            text = clean(a.get_text(" ", strip=True))

            if not text:
                continue

            if any(k in text.lower() for k in ["quán", "hải sản", "đà nẵng"]):
                if len(text.split()) <= 10:
                    related.append(text)

        related = list(dict.fromkeys(related))[:10]

        # =========================
        # 4. SNIPPETS (clean SERP preview)
        # =========================
        snippets = []

        for block in soup.select("div.g"):
            snippet_el = block.select_one("span")

            if not snippet_el:
                continue

            text = clean(snippet_el.get_text(" ", strip=True))

            if text and 60 < len(text) < 300:
                snippets.append({
                    "snippet": text[:200]
                })

        snippets = snippets[:5]

        # =========================
        # 5. KEYWORD CLUSTER
        # =========================
        keyword_cluster = [
            "quán hải sản ngon Đà Nẵng",
            "hải sản giá rẻ",
            "gần biển Mỹ Khê",
            "Sơn Trà",
            "quán nhậu Đà Nẵng",
            "buffet hải sản"
        ]

        # =========================
        # 6. CONTENT ANGLE
        # =========================
        content_angle = [
            "review",
            "top list",
            "trải nghiệm",
            "giá rẻ",
            "du lịch ăn uống"
        ]

        # =========================
        # 7. INTENT LAYER
        # =========================
        intent = [
            "ăn nhậu",
            "du lịch",
            "gia đình",
            "buffet",
            "hải sản tươi sống"
        ]

        # =========================
        # 8. COMPETITOR PATTERN
        # =========================
        competitor_pattern = [
            "Pasgo",
            "Tripadvisor",
            "Blog du lịch SEO",
            "Facebook group",
            "Local listing / directory"
        ]

        # =========================
        # FINAL OUTPUT
        # =========================
        return {
            "query": QUERY,
            "top_urls": top_urls[:10],
            "people_also_ask": paa,
            "related_searches": related,
            "snippets": snippets,
            "keyword_cluster": keyword_cluster,
            "content_angle": content_angle,
            "intent": intent,
            "competitor_pattern": competitor_pattern
        }

    finally:
        if browser:
            browser.stop()


# =========================
# JUPYTER RUN
# =========================
node1 = await node1_serp()
from pprint import pprint
pprint(node1)

{'competitor_pattern': ['Pasgo',
                        'Tripadvisor',
                        'Blog du lịch SEO',
                        'Facebook group',
                        'Local listing / directory'],
 'content_angle': ['review',
                   'top list',
                   'trải nghiệm',
                   'giá rẻ',
                   'du lịch ăn uống'],
 'intent': ['ăn nhậu', 'du lịch', 'gia đình', 'buffet', 'hải sản tươi sống'],
 'keyword_cluster': ['quán hải sản ngon Đà Nẵng',
                     'hải sản giá rẻ',
                     'gần biển Mỹ Khê',
                     'Sơn Trà',
                     'quán nhậu Đà Nẵng',
                     'buffet hải sản'],
 'people_also_ask': [],
 'query': 'nhà hàng quán nhậu hải sản Đà Nẵng',
 'related_searches': ['AEON MALL Đà Nẵng Thanh Khê',
                      'Quán Eo Biển',
                      'Nhà hàng hải sản Bé Anh...',
                      'Nhà hàng hải sản Cua Biển...',
                      'Nhà hàng hải 

In [1]:
#============= OKOK==========

import asyncio
import random
import traceback
import nodriver as uc

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"


async def inspect_html_and_click():
    browser = None

    try:
        print("🚀 Mở trình duyệt...")

        browser = await uc.start(
            headless=False,
            icon_mode=1
        )

        print("🔍 Vào Google...")

        page = await browser.get(
            "https://www.google.com/search?q=site:facebook.com/mocseafood"
        )

        print("⏳ Đợi kết quả xuất hiện...")

        target_element = None

        for _ in range(30):
            try:
                target_element = await page.select(
                    f'a[href="{TARGET_PAGE_URL}"]'
                )

                if target_element:
                    break

            except Exception:
                pass

            await asyncio.sleep(0.2)

        if not target_element:
            print("❌ Không tìm thấy link.")
            return

        print("✅ Đã tìm thấy link.")

        try:
            await target_element.scroll_into_view()
        except Exception:
            pass

        await asyncio.sleep(
            random.uniform(0.34, 0.8)
        )

        print("🖱️ Click Facebook...")

        await target_element.click()

        await asyncio.sleep(
            random.uniform(1.5, 2.5)
        )

        print("🔎 Kiểm tra tabs...")

        fb_tab = None

        for tab in browser.tabs:
            try:
                url = str(tab.url)

                print("TAB:", url)

                if "facebook.com" in url.lower():
                    fb_tab = tab

            except Exception:
                pass

        if not fb_tab:
            print("❌ Không tìm thấy tab Facebook.")
            return

        print("🎉 Đã tìm thấy Facebook.")

        print("📜 Scroll + Expand Posts + Expand Comments...")
        
        for round_idx in range(2):
        
            print(f"🔄 Round {round_idx + 1}/10")
        
            # =====================================================
            # SCROLL
            # =====================================================
        
            try:
                await fb_tab.evaluate(
                    """
                    window.scrollBy(0, 3000);
                    """
                )
            except Exception:
                pass
        
            await asyncio.sleep(
                random.uniform(1.5, 2.5)
            )

        print("📸 Lấy HTML...")
    
        html = await fb_tab.get_content()

        with open(
            "facebook_page.html",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(html)

        print(
            f"💾 Đã lưu facebook_page.html "
            f"({len(html):,} ký tự)"
        )
    
        # =====================================================
        # CLICK "SEE MORE"
        # CLICK "VIEW MORE COMMENTS"
        # =====================================================
        
        try:
    
            await fb_tab.evaluate(
                """
                (() => {
    
                    const targets = [
                        "See more",
                        "View more comments",
                        "View previous comments",
                        "View more replies",
    
                        "Xem thêm",
                        "Xem thêm bình luận",
                        "Xem phản hồi",
                        "Xem thêm phản hồi"
                    ];
    
                    const all = document.querySelectorAll("*");
    
                    for (const el of all) {
    
                        const text =
                            (el.innerText || "")
                            .trim();
    
                        if (
                            targets.includes(text)
                        ) {
    
                            try {
                                el.click();
                            } catch(e) {}
    
                        }
                    }
    
                })();
                """
            )
    
        except Exception:
            pass
    
        await asyncio.sleep(
            random.uniform(0.6, 1.2)
        )
        
        print("✅ Hoàn tất expand.")
        # ==========================================
        # DEBUG DIALOG
        # ==========================================
        await asyncio.sleep(2)
        try:
        
            dialog_count = await fb_tab.evaluate("""
            document.querySelectorAll('[role="dialog"]').length
            """)
        
            print("FOUND DIALOGS:", dialog_count)
        
            popup_html = await fb_tab.evaluate("""
            (() => {
                const dialogs =
                    document.querySelectorAll('[role="dialog"]');
        
                if (!dialogs.length)
                    return "NO_DIALOG";
        
                return dialogs[dialogs.length - 1].outerHTML;
            })()
            """)
        
            print("POPUP SIZE:", len(popup_html))
        
            with open(
                "popup_only.html",
                "w",
                encoding="utf-8"
            ) as f:
                f.write(popup_html)
        
            print("💾 Saved popup_only.html")
        
        except Exception as e:
        
            print("Popup debug error:", e)
        

    except Exception as e:
        print("❌ Lỗi:", e)
        traceback.print_exc()

    finally:
        print("⏳ Giữ trình duyệt 30s...")
        #await asyncio.sleep(1)

        try:
            if browser:
                browser.stop()
        except Exception:
            pass
            
if __name__ == "__main__":
    await inspect_html_and_click()

🚀 Mở trình duyệt...
🔍 Vào Google...
⏳ Đợi kết quả xuất hiện...
✅ Đã tìm thấy link.
🖱️ Click Facebook...
🔎 Kiểm tra tabs...
TAB: https://www.google.com/search?q=site:facebook.com/mocseafood&sei=PzU1atGxNMeNvr0PoYTUuA0
🎉 Đã tìm thấy Facebook.
📜 Scroll + Expand Posts + Expand Comments...
🔄 Round 1/10
🔄 Round 2/10
📸 Lấy HTML...
💾 Đã lưu facebook_page.html (1,765,503 ký tự)
✅ Hoàn tất expand.
Popup debug error: no close frame received or sent
⏳ Giữ trình duyệt 30s...


In [1]:
from bs4 import BeautifulSoup
import re
import json

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "facebook_page.html"
OUTPUT_FILE = "brand_raw.json"

# ============================================================
# LOAD HTML
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

raw_text = soup.get_text(
    "\n",
    strip=True
)

# ============================================================
# CLEAN FACEBOOK NOISE
# ============================================================

NOISE_LINES = {
    "Like",
    "Comment",
    "Share",
    "See more",
    "View more comments",
    "All reactions:",
    "Online status indicator",
    "Active Status indicator",
    "Privacy",
    "Terms",
    "Advertising",
    "Ad choices",
    "Cookies",
    "More",
    "Active",
    "Log in",
    "Forgotten account?",
    "Chia sẻ",
    "Thích",
    "Bình luận"
}

clean_lines = []

for line in raw_text.split("\n"):

    line = line.strip()

    if not line:
        continue

    if line in NOISE_LINES:
        continue

    # Chỉ lọc số lượng phản hồi nếu đứng độc lập (Ví dụ dòng chỉ có số "94")
    if re.fullmatch(r"\d+", line):
        continue

    # Lọc thời gian video độc lập
    if re.fullmatch(r"\d+:\d+", line):
        continue

    clean_lines.append(line)

clean_text = "\n".join(clean_lines)

# ============================================================
# PAGE INFO
# ============================================================

page_info = {}

title_tag = soup.find("title")

if title_tag:
    page_info["title"] = title_tag.get_text(strip=True)

# Sửa regex nhận diện được cả chữ tiếng Việt "người theo dõi" hoặc "followers"
m = re.search(
    r'([\d.,]+\s*[KMB]?)\s+(?:followers|người theo dõi)',
    clean_text,
    re.IGNORECASE
)
if m:
    page_info["followers"] = m.group(1).strip()

m = re.search(
    r'([\d.,]+\s*[KMB]?)\s+(?:following|đang theo dõi)',
    clean_text,
    re.IGNORECASE
)
if m:
    page_info["following"] = m.group(1).strip()

# ============================================================
# INTRO SECTION (Gom trọn vẹn cụm thông tin)
# ============================================================

intro_section = ""

# Tìm vị trí bắt đầu linh hoạt dựa trên giao diện hiển thị
start_pos = -1
for marker in ["Intro", "Giới thiệu", "Page ·"]:
    pos = clean_text.find(marker)
    if pos != -1:
        start_pos = pos
        break

if start_pos != -1:
    end_pos = len(clean_text)
    # Cắt chính xác đến trước khi danh sách Ảnh hoặc Bài viết bắt đầu để ôm trọn data
    for marker in ["\nPhotos", "\nPosts", "\nAbout", "\nẢnh", "\nBài viết"]:
        pos = clean_text.find(marker, start_pos)
        if pos != -1 and pos < end_pos:
            end_pos = pos
            
    intro_section = clean_text[start_pos:end_pos].strip()
else:
    intro_section = clean_text[:2000].strip()

# ============================================================
# STRUCTURED DATA (Trích xuất bổ sung để đối chiếu khi cần)
# ============================================================

# Phones
raw_phones = re.findall(
    r'(?:\+84\s?)?(?:0\d{2,3}[\s.-]?\d{3}[\s.-]?\d{3,4})',
    clean_text
)

phones = set()
for phone in raw_phones:
    phone = re.sub(r"[\s.-]", "", phone)
    if phone.startswith("+84"):
        phone = "0" + phone[3:]
    phones.add(phone)
phones = sorted(list(phones))

# Emails
emails = sorted(set(
    re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', clean_text)
))

# Domains (Loại trừ phần mở rộng file tĩnh hệ thống)
all_domains = re.findall(r'\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,6}\b', clean_text.lower())
domains = sorted(list(set([d for d in all_domains if not d.endswith(('js', 'css', 'png', 'jpg', 'jpeg', 'gif', 'ico'))])))

# ============================================================
# POSTS
# ============================================================

posts = []
chunks = re.split(r'Shared with Public|Chia sẻ với Công khai|Công khai', clean_text)

for chunk in chunks:
    chunk = chunk.strip()
    if len(chunk) < 150:
        continue
    if "followers" in chunk or "người theo dõi" in chunk:
        continue
    posts.append(chunk)

posts = posts[:10]

# ============================================================
# RESULT (Cấu trúc chuẩn, vừa phải, info giữ 100% data)
# ============================================================

result = {
    "page_info": page_info,
    "info": intro_section,
    "extracted_details": {
        "phones": phones,
        "emails": emails,
        "domains": domains,
        "recent_posts": posts
    }
}

# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 80)
print("FINAL JSON OUTPUT PREVIEW")
print("=" * 80)
print(json.dumps(result, ensure_ascii=False, indent=2)[:1500] + "\n...")
print(f"\n💾 Saved -> {OUTPUT_FILE}")

FINAL JSON OUTPUT PREVIEW
{
  "page_info": {
    "title": "Hải sản Mộc quán Đà Nẵng | Da Nang  | Facebook",
    "followers": "43K"
  },
  "info": "Intro\n⏰ Giờ mở cửa: 10:30AM – 23:45PM\n(Nhận order cuối lúc 22:30PM)\n📍 Cơ sở 1: 26 Tô Hiến Thành, Sơn Trà, Đà Nẵng\nHotline Đà Nẵng: 0964 531 153 – 0905 665 058\n📍 Cơ sở 2: 74–76 Hồng Bàng, Tân Lập, Nha Trang\nHotline Nha Trang: 0934 981 180\nPage\n· Seafood restaurant\n26 Tô Hiến Thành, Da Nang, Vietnam\n090 566 50 58\n+84 90 566 50 58\nmocseafood@facebook.com\nhaisanmocquandanang\nmocseafood_dn\nhaisanmocquan\nmocseafood_dn\n다낭목해산물식당\nmocseafood.com\nguide.michelin.com/vn/en/da-nang-region/da-nang_2984390/restaurant/moc\ntripadvisor.com/Restaurant_Review-g298085-d16891168-Reviews-M_c_Seafood_Restaurant-Da_Nang.html\nOpen now\nDelivery · Takeaway · Dine in…\nPrice range · ££\n94% recommend (79 reviews)\n﻿",
  "extracted_details": {
    "phones": [
      "0905665058",
      "0934981180",
      "0964531153"
    ],
    "emails": [
      "moc

In [2]:
# ============= OK OKOK ========================
import json
import re
from bs4 import BeautifulSoup

def clean_fb_text(text):
    """Hàm sửa lỗi mã hóa font chữ của Facebook (UTF-8 thành Latin-1)"""
    if not text:
        return ""
    try:
        return text.encode('latin1').decode('utf-8')
    except Exception:
        return text

# 1. Đọc tệp HTML
html_file_path = "popup_only.html"
with open(html_file_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

# 2. Thu thập nội dung bài viết chính (Post Content)
post_contents = []
# Facebook thường để nội dung bài viết trong các thẻ div có dir="auto" hoặc lớp text cụ thể
for post_div in soup.find_all("div", attrs={"data-ad-preview": "message"}):
    post_contents.append(clean_fb_text(post_div.get_text(separator="\n", strip=True)))

# Nếu không tìm thấy bằng data-ad-preview, quét diện rộng các khối div văn bản chính
if not post_contents:
    for text_div in soup.find_all("div", class_="xdj266r"):
        # Chỉ lấy các khối có văn bản dài và chưa bị lồng vào phần comment
        if len(text_div.get_text()) > 100 and not text_div.find_parent(attrs={"role": "article"}):
            cleaned = clean_fb_text(text_div.get_text(strip=True))
            if cleaned and cleaned not in post_contents:
                post_contents.append(cleaned)

# 3. Thu thập toàn bộ Bình luận (Comments) và Phản hồi (Replies)
comments_list = []
comment_blocks = soup.find_all("div", attrs={"role": "article"})

for block in comment_blocks:
    aria_label = block.get("aria-label", "")
    
    # Tìm tên người bình luận và thời gian từ aria-label (Ví dụ: "Comment by Dung Thai Ngoc 2 days ago")
    author_name = "Ẩn danh"
    comment_time = "Không rõ"
    if "Comment by" in aria_label:
        meta_info = aria_label.replace("Comment by ", "")
        # Tách tên và thời gian tương đối bằng Regex nếu có thể
        match = re.search(r'(.+?)\s(\d+\s\w+\sago|\d+\w+)', meta_info)
        if match:
            author_name = match.group(1)
            comment_time = match.group(2)
        else:
            author_name = meta_info

    # Lấy nội dung văn bản của bình luận
    comment_text = ""
    text_container = block.find("div", attrs={"dir": "auto"})
    if text_container:
        comment_text = clean_fb_text(text_container.get_text(strip=True))
    else:
        # Cách tìm dự phòng nếu không thấy thẻ dir="auto"
        possible_text = block.find("div", class_="xdj266r")
        if possible_text:
            comment_text = clean_fb_text(possible_text.get_text(strip=True))

    # Tìm phản hồi (Replies) đi kèm ngay dưới bình luận này
    replies = []
    parent_div = block.find_parent()
    if parent_div:
        next_sibling = parent_div.find_next_sibling()
        if next_sibling:
            # Nếu khối tiếp theo chứa thông tin "replied" hoặc các comment con
            reply_text = clean_fb_text(next_sibling.get_text(separator=" ", strip=True))
            if reply_text:
                replies.append(reply_text)

    # Chỉ lưu nếu bình luận có nội dung thực tế
    if comment_text or author_name != "Ẩn danh":
        comments_list.append({
            "author": author_name,
            "time": comment_time,
            "comment": comment_text,
            "replies": replies
        })

# 4. Gom tất cả dữ liệu vào một cấu trúc JSON lớn
final_data = {
    "source_file": html_file_path,
    "total_comments_found": len(comments_list),
    "main_post_data": post_contents,
    "detailed_comments": comments_list
}

# 5. Xuất ra file JSON đầy đủ
output_json_path = "facebook_detailed_data.json"
with open(output_json_path, "w", encoding="utf-8") as json_file:
    json.dump(final_data, json_file, ensure_ascii=False, indent=4)

print(f"--- ĐÃ TRÍCH XUẤT XONG ---")
print(f"Tìm thấy {len(post_contents)} đoạn bài viết chính.")
print(f"Tìm thấy {len(comments_list)} bình luận chi tiết.")
print(f"File kết quả đã lưu tại: {output_json_path}")

--- ĐÃ TRÍCH XUẤT XONG ---
Tìm thấy 11 đoạn bài viết chính.
Tìm thấy 34 bình luận chi tiết.
File kết quả đã lưu tại: facebook_detailed_data.json


In [1]:
# ================================================================
# RESEARCH WORKFLOW — LANGGRAPH
# 4 Nodes: Suggest → SERP → FB Scrape → FB Parse
# ================================================================

import asyncio, json, re, time, random, traceback
from typing import TypedDict, Optional
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup
import nodriver as uc

from langgraph.graph import StateGraph, END


# ================================================================
# STATE
# ================================================================

class ResearchState(TypedDict):
    query: str
    fb_url: str

    # Node 1
    suggestions: list
    tagged: dict

    # Node 2
    serp: dict

    # Node 3
    fb_page_html: str
    fb_popup_html: str

    # Node 4
    brand: dict
    comments: dict

    # Final
    report: dict


# ================================================================
# NODE 1 — GOOGLE SUGGEST COLLECTOR
# ================================================================

def _suggest(query: str) -> list:
    url = "https://suggestqueries.google.com/complete/search"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/109.0",
        "Accept-Language": "vi-VN,vi;q=0.9",
        "Referer": "https://www.google.com/"
    }
    try:
        r = requests.get(url, params={"client": "firefox", "hl": "vi", "q": query},
                         headers=headers, timeout=15)
        data = r.json()
        if isinstance(data, list) and len(data) >= 2:
            return [s for s in data[1] if s != query and len(s) > 3]
    except Exception:
        pass
    return []


def _intent_tag(queries: list) -> dict:
    tags = {
        "budget":     ["giá", "rẻ", "tiết kiệm", "bao nhiêu", "chi phí"],
        "location":   ["gần", "ở", "tại", "đường", "khu"],
        "food":       ["hải sản", "món", "ăn", "nhậu", "buffet", "nướng"],
        "experience": ["review", "trải nghiệm", "ngon", "đẹp", "view"],
        "family":     ["gia đình", "nhóm", "đoàn", "nhiều người"],
        "question":   ["?", "là gì", "ở đâu", "có nên", "nào"],
    }
    result = {t: [] for t in tags}
    result["other"] = []
    for q in queries:
        q_low = q.lower()
        matched = False
        for tag, kws in tags.items():
            if any(k in q_low for k in kws):
                result[tag].append(q)
                matched = True
                break
        if not matched:
            result["other"].append(q)
    return result


def node_suggest(state: ResearchState) -> ResearchState:
    """Node 1: Thu thập Google Suggest → phân loại intent."""
    query = state["query"]
    pool = set(_suggest(query))

    # Mở rộng: alphabet + số + intent words
    for char in "abcdefghijklmnopqrstuvwxyz0123456789":
        pool.update(_suggest(f"{query} {char}"))
        time.sleep(0.3)

    for word in ["giá", "gần", "ở", "cho", "review", "nào", "ngon", "rẻ", "đẹp"]:
        pool.update(_suggest(f"{query} {word}"))
        time.sleep(0.3)

    suggestions = list(pool)
    return {**state, "suggestions": suggestions, "tagged": _intent_tag(suggestions)}


# ================================================================
# NODE 2 — SERP SCRAPER
# ================================================================

def _clean_text(text: str) -> Optional[str]:
    if not text:
        return None
    text = " ".join(text.split())
    BAD = ["facebook", "tripadvisor", "đăng nhập", "chính sách",
           "youtube", "maps", "google", "javascript", "cookie"]
    if len(text) < 5 or len(text) > 160 or any(x in text.lower() for x in BAD):
        return None
    return text


async def node_serp(state: ResearchState) -> ResearchState:
    """Node 2: Scrape Google SERP — organic URLs, PAA, snippets, related."""
    query = state["query"]
    browser = await uc.start(headless=False)
    try:
        page = await browser.get(f"https://www.google.com/search?q={quote(query)}&hl=vi")
        await asyncio.sleep(3)
        soup = BeautifulSoup(await page.get_content(), "html.parser")

        # Organic URLs
        top_urls, seen = [], set()
        for a in soup.select("a"):
            h3 = a.select_one("h3")
            if not h3:
                continue
            url = a.get("href", "")
            title = _clean_text(h3.get_text(" ", strip=True))
            if not title or not url.startswith("http"):
                continue
            key = url.split("#")[0]
            if key in seen:
                continue
            seen.add(key)
            top_urls.append({"position": len(top_urls) + 1, "title": title,
                              "url": url, "domain": url.split("/")[2]})

        # People Also Ask
        paa = []
        for el in soup.select('div[jsname="N760b"]'):
            q = _clean_text(el.get_text(" ", strip=True))
            if q and "?" in q:
                paa.append(q)

        # Snippets
        snippets = []
        for block in soup.select("div.g"):
            el = block.select_one("span")
            if not el:
                continue
            t = _clean_text(el.get_text(" ", strip=True))
            if t and 60 < len(t) < 300:
                snippets.append(t[:200])

        serp = {
            "query": query,
            "top_urls": top_urls[:10],
            "people_also_ask": paa[:5],
            "snippets": snippets[:5],
        }
    finally:
        browser.stop()

    return {**state, "serp": serp}



### node 3




async def node_fb(state):
    """
    Node 3+4 gom lại:
    Scrape Facebook → parse ngay trong memory → chỉ trả data sạch vào state.
    Không lưu HTML, không lưu file tạm, không đưa raw HTML vào state.
    """
    fb_url = state["fb_url"]
    slug   = fb_url.rstrip("/").split("/")[-1]

    browser  = None
    brand    = {}
    comments = {}

    try:
        print("🚀 Mở trình duyệt...")
        browser = await uc.start(headless=False, icon_mode=1)

        # ── Google → tìm link FB ──
        print("🔍 Vào Google...")
        page = await browser.get(
            f"https://www.google.com/search?q=site:facebook.com/{slug}"
        )

        target_element = None
        for _ in range(30):
            try:
                target_element = await page.select(f'a[href="{fb_url}"]')
                if target_element:
                    break
            except Exception:
                pass
            await asyncio.sleep(0.2)

        if not target_element:
            print("❌ Không tìm thấy link.")
            return {**state, "brand": {}, "comments": {}}

        print("✅ Tìm thấy link → Click...")
        try:
            await target_element.scroll_into_view()
        except Exception:
            pass
        await asyncio.sleep(random.uniform(0.34, 0.8))
        await target_element.click()
        await asyncio.sleep(random.uniform(1.5, 2.5))

        # ── Tìm tab Facebook ──
        fb_tab = None
        for tab in browser.tabs:
            try:
                if "facebook.com" in str(tab.url).lower():
                    fb_tab = tab
                    print(f"🎉 FB tab: {tab.url}")
            except Exception:
                pass

        if not fb_tab:
            print("❌ Không tìm thấy tab Facebook.")
            return {**state, "brand": {}, "comments": {}}

        # ── Scroll để load content ──
        for i in range(3):
            print(f"🔄 Scroll {i+1}/2")
            try:
                await fb_tab.evaluate("window.scrollBy(0, 3000);")
            except Exception:
                pass
            await asyncio.sleep(random.uniform(1.5, 2.5))

        # ── Lấy HTML page → parse brand ngay ──
        print("📸 Lấy page HTML → parse brand...")
        page_html = await fb_tab.get_content()
        brand = _parse_brand(page_html)          # parse ngay, bỏ HTML
        print(f"   title={brand.get('page_info',{}).get('title','?')} | posts={len(brand.get('posts',[]))}")

        # ── Expand comments ──
        try:
            await fb_tab.evaluate("""
                (() => {
                    const targets = [
                        "See more", "View more comments",
                        "View previous comments", "View more replies",
                        "Xem thêm", "Xem thêm bình luận",
                        "Xem phản hồi", "Xem thêm phản hồi"
                    ];
                    for (const el of document.querySelectorAll("*")) {
                        if (targets.includes((el.innerText || "").trim())) {
                            try { el.click(); } catch(e) {}
                        }
                    }
                })();
            """)
        except Exception:
            pass
        await asyncio.sleep(random.uniform(0.6, 1.2))

        # ── Lấy popup → parse comments ngay ──
        print("💬 Lấy popup HTML → parse comments...")
        await asyncio.sleep(3)
        popup_html = ""
        try:
            dialog_count = await fb_tab.evaluate(
                "document.querySelectorAll('[role=\"dialog\"]').length"
            )
            print(f"   dialogs found: {dialog_count}")
            if dialog_count:
                popup_html = await fb_tab.evaluate("""
                    (() => {
                        const d = document.querySelectorAll('[role="dialog"]');
                        return d[d.length - 1].outerHTML;
                    })()
                """)
        except Exception as e:
            print(f"   Popup error: {e}")

        # Fallback: không có popup thì dùng page HTML cho comments
        comments = _parse_comments(popup_html or page_html)
        print(f"   comments={comments.get('total', 0)}")

    except Exception as e:
        print("❌ Lỗi:", e)
        traceback.print_exc()

    finally:
        try:
            if browser:
                browser.stop()
        except Exception:
            pass

    return {"brand": brand, "comments": comments}


def _parse_comments(html: str) -> dict:
    """Parse comments + replies từ popup_only.html."""
    if not html:
        return {"total": 0, "post_content": [], "comments": []}

    soup = BeautifulSoup(html, "html.parser")

    # Bài viết chính
    post_contents = []
    for div in soup.find_all("div", attrs={"data-ad-preview": "message"}):
        post_contents.append(_fix_encoding(div.get_text(separator="\n", strip=True)))
    if not post_contents:
        for div in soup.find_all("div", class_="xdj266r"):
            t = _fix_encoding(div.get_text(strip=True))
            if len(t) > 100 and t not in post_contents:
                post_contents.append(t)

    # Comments
    comments = []
    for block in soup.find_all("div", attrs={"role": "article"}):
        label = block.get("aria-label", "")
        author, ctime = "Ẩn danh", "Không rõ"
        if "Comment by" in label:
            info = label.replace("Comment by ", "")
            m = re.search(r'(.+?)\s(\d+\s\w+\sago|\d+\w+)', info)
            if m:
                author, ctime = m.group(1), m.group(2)
            else:
                author = info

        text_el = block.find("div", attrs={"dir": "auto"}) or block.find("div", class_="xdj266r")
        comment_text = _fix_encoding(text_el.get_text(strip=True)) if text_el else ""

        if comment_text or author != "Ẩn danh":
            comments.append({"author": author, "time": ctime, "comment": comment_text})

    return {"total": len(comments), "post_content": post_contents, "comments": comments}


def node_fb_parse(state: ResearchState) -> ResearchState:
    """Node 4: Parse HTML → brand info + comments."""
    brand   = _parse_brand(state["fb_page_html"])
    comments = _parse_comments(state["fb_popup_html"])
    return {**state, "brand": brand, "comments": comments}


# ================================================================
# FINAL — BUILD REPORT
# ================================================================

def node_report(state: ResearchState) -> ResearchState:
    """Tổng hợp toàn bộ output thành 1 research report."""
    report = {
        "query":       state["query"],
        "suggestions": {"total": len(state["suggestions"]), "tagged": state["tagged"]},
        "serp":        state["serp"],
        "brand":       state["brand"],
        "comments":    state["comments"],
    }
    return {**state, "report": report}



In [22]:
test_state = {
    "query":    "nhà hàng quán nhậu hải sản Đà Nẵng",
    "fb_url":   "https://www.facebook.com/mocseafood/",
    "brand":    {},
    "comments": {},
}
 
result = await node_fb(test_state)

brand    = result["brand"]
comments = result["comments"]
print(brand)
print("===========================================")
print(comments)


🚀 Mở trình duyệt...
🔍 Vào Google...
✅ Tìm thấy link → Click...
🎉 FB tab: https://www.google.com/search?q=site:facebook.com/mocseafood&sei=5co0atrJLdSKvr0Py-DusAU
🔄 Scroll 1/2
🔄 Scroll 2/2
🔄 Scroll 3/2
📸 Lấy page HTML → parse brand...
   title=Hải sản Mộc quán Đà Nẵng | Da Nang  | Facebook | posts=10
💬 Lấy popup HTML → parse comments...
   dialogs found: 2
   comments=67
{'page_info': {'title': 'Hải sản Mộc quán Đà Nẵng | Da Nang  | Facebook', 'followers': '43K'}, 'intro': 'Intro\n⏰ Giờ mở cửa: 10:30AM – 23:45PM\n(Nhận order cuối lúc 22:30PM)\n📍 Cơ sở 1: 26 Tô Hiến Thành, Sơn Trà, Đà Nẵng\nHotline Đà Nẵng: 0964 531 153 – 0905 665 058\n📍 Cơ sở 2: 74–76 Hồng Bàng, Tân Lập, Nha Trang\nHotline Nha Trang: 0934 981 180\nPage\n· Seafood restaurant\n26 Tô Hiến Thành, Da Nang, Vietnam\n090 566 50 58\n+84 90 566 50 58\nmocseafood@facebook.com\nhaisanmocquandanang\nmocseafood_dn\nhaisanmocquan\nmocseafood_dn\n다낭목해산물식당\nmocseafood.com\nguide.michelin.com/vn/en/da-nang-region/da-nang_2984390/restaur

In [3]:
import asyncio
import random
import traceback
import nodriver as uc


async def _scrape_facebook(fb_url: str, headless: bool, business_id: str):
    """
    HÀM CÀO DỮ LIỆU FACEBOOK - ĐÃ ÉP CHẠY CHẬM TUẦN TỰ, ĐỢI 3S SAU CLICK
    """
    browser = None

    try:
        print("🚀 Mở trình duyệt...")
        browser = await uc.start(headless=headless, icon_mode=1)

        # 1. Chuẩn hóa URL mục tiêu (Luôn có dấu / ở cuối)
        target_url = fb_url.strip().lower()
        if not target_url.endswith("/"):
            target_url += "/"

        # Ép chuỗi có cả dấu ngoặc kép để hiển thị và so sánh như bạn muốn
        clean_search_url = f'"{target_url}"'  # '"https://www.facebook.com/mocseafood/"'

        print(f"🔍 Vào Google tìm kiếm từ khóa link trực tiếp: {clean_search_url}...")
        page = await browser.get(f"https://www.google.com/search?q={clean_search_url}")

        # Thêm thời gian đợi sau khi vào Google cho trang load thật ổn định
        print("⏳ Chờ 3 giây cho trang Google Search tải xong hoàn toàn...")
        await asyncio.sleep(3.0)

        matched_href = None
        click_target = None  # Biến lưu trữ Element dùng để click

        print("\n🔄 Bắt đầu quét các kết quả hiển thị...")
        links = await page.select_all("a:has(h3)")

        if not links:
            print("⚠️ Không tìm thấy kết quả nào hiển thị trên Google!")
            return None, None

        print(f"📊 --- XUẤT TOÀN BỘ LOG {len(links)} KẾT QUẢ TÌM THẤY ---")

        for i, link in enumerate(links):
            try:
                href_attr = link["href"] if "href" in link.attrs else None
                text_content = link.text.strip().replace('\n', ' ') if link.text else "KHÔNG CÓ TEXT"
                class_attr = link["class"] if "class" in link.attrs else "No Class"

                print(f"🔹 Kết quả [{i}]:")
                print(f"   + Text tiêu đề : {text_content}")
                print(f"   + Class thẻ <a>: {class_attr}")
                print(f"   + Link gốc URL : {href_attr}")

                if not href_attr:
                    print("   ❌ Trạng thái  : Bỏ qua (Không có href)")
                    print("-" * 60)
                    continue
                    
                href_lower = href_attr.strip().lower()
                # Bọc dấu ngoặc kép vào href quét được để so khớp chính xác tuyệt đối với clean_search_url
                href_with_quotes = f'"{href_lower}"'

                # SO SÁNH CHUẨN XÁC TẠI ĐÂY
                if clean_search_url in href_with_quotes:
                    print(f"   🎯 Trạng thái  : ✅ TRÙNG KHỚP HOÀN HẢO!")
                    if not matched_href:
                        matched_href = href_attr.strip()
                        click_target = link  # Lưu lại Element chuẩn để tí nữa click
                else:
                    print("   ❌ Trạng thái  : Không khớp từ khóa")

                print("-" * 60)

            except Exception as e:
                print(f"   ❌ Lỗi đọc phần tử ở vị trí [{i}]: {e}")
                print("-" * 60)

        print(f"📊 --- KẾT THÚC DANH SÁCH LOG --- \n")

        if not matched_href or not click_target:
            print(f"❌ Không tìm thấy homepage khớp với từ khóa: {fb_url}")
            return None, None

        # =====================================================
        # ⚡ BẺ HƯỚNG CHỐNG NHẢY TAB MỚI TRƯỚC KHI CLICK
        # =====================================================
        print(f"🪄 Tiến hành bẻ hướng target='_blank' của link chuẩn trên giao diện...")
        try:
            await page.evaluate(
                f"""
                (() => {{
                    const el = document.querySelector('a[href="{matched_href}"]');
                    if (el) {{
                        el.removeAttribute('target');
                        el.setAttribute('rel', 'noopener noreferrer');
                    }}
                }})();
                """
            )
        except Exception as e:
            print(f"⚠️ Cảnh báo lỗi JS xử lý thuộc tính target: {e}")

        # Chậm lại một chút trước khi bấm chuột
        await asyncio.sleep(1.0)

        try:
            await click_target.scroll_into_view()
        except Exception:
            pass

        print(f"🖱️ [HÀNH ĐỘNG] Click vào kết quả Facebook: {matched_href}")
        await click_target.click()

        # =====================================================
        # ĐỢI 3 GIÂY CỐ ĐỊNH THEO YÊU CẦU CỦA BẠN
        # =====================================================
        print("⏳ Đang đợi 3 giây cố định sau khi click để trang chuyển hướng mượt mà...")
        await asyncio.sleep(3.0)

        # Chờ thêm một khoảng nhỏ dự phòng cho Facebook load xong giao diện chính
        print("⏳ Đợi thêm 3.5 giây cho Facebook nạp toàn bộ bài viết...")
        await asyncio.sleep(3.5)

        print("🔎 Kiểm tra trang Facebook...")
        fb_tab = page 
        current_url = str(fb_tab.url).lower()
        print(f"🔗 URL hiện tại sau khi điều hướng: {current_url}")

        if "login" in current_url or "checkpoint" in current_url:
            print("❌ Thất bại: Facebook chặn checkpoint / bắt Login rồi!")
            return None, None

        print("🎉 Đã vào được Facebook hợp lệ.")
        print("📜 Tiến hành cuộn trang (Scroll)...")

        for round_idx in range(2):
            print(f"🔄 Round {round_idx + 1}/2")
            try:
                await fb_tab.evaluate("window.scrollBy(0, 3000);")
            except Exception:
                pass
            await asyncio.sleep(2.0)

        print("📸 Lấy dữ liệu HTML...")
        html = await fb_tab.get_content()
        page_path = f"fb_page_{business_id}.html"

        with open(page_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"💾 Đã lưu {page_path} ({len(html):,} ký tự)")
        return page_path, None

    except Exception as e:
        print("❌ Lỗi toàn cục hàm Facebook:", e)
        traceback.print_exc()
        return None, None

    finally:
        print("⏳ Đóng trình duyệt...")
        try:
            if browser:
                await browser.stop()
                await asyncio.sleep(0.5)  # Tránh lỗi Event loop is closed
        except Exception:
            pass

async def main():
    fb_url = "https://www.facebook.com/mocseafood"
    print(f"🔥 Bắt đầu test hàm với URL: {fb_url}")
    
    page, popup = await _scrape_facebook(
        fb_url=fb_url, 
        headless=False,       
        business_id="test_id_001"
    )
    
    print("\n==============================")
    print(f"🎉 KẾT QUẢ TRẢ VỀ:\n- Page HTML Path: {page}")
    print("==============================")

if __name__ == "__main__":
    await main()

🔥 Bắt đầu test hàm với URL: https://www.facebook.com/mocseafood
🚀 Mở trình duyệt...
🔍 Vào Google tìm kiếm từ khóa link trực tiếp: "https://www.facebook.com/mocseafood/"...
⏳ Chờ 3 giây cho trang Google Search tải xong hoàn toàn...

🔄 Bắt đầu quét các kết quả hiển thị...
📊 --- XUẤT TOÀN BỘ LOG 8 KẾT QUẢ TÌM THẤY ---
🔹 Kết quả [0]:
   + Text tiêu đề : Hải sản Mộc quán Đà Nẵng | Da Nang
   + Class thẻ <a>: No Class
   + Link gốc URL : https://www.facebook.com/mocseafood/
   🎯 Trạng thái  : ✅ TRÙNG KHỚP HOÀN HẢO!
------------------------------------------------------------
🔹 Kết quả [1]:
   + Text tiêu đề : Hải sản Mộc quán Đà Nẵng (@mocseafood) - Home
   + Class thẻ <a>: No Class
   + Link gốc URL : https://www.facebook.com/mocseafood/services/
   ❌ Trạng thái  : Không khớp từ khóa
------------------------------------------------------------
🔹 Kết quả [2]:
   + Text tiêu đề : Hải sản Mộc quán Đà Nẵng (@mocseafood) - Photos
   + Class thẻ <a>: No Class
   + Link gốc URL : https://www.faceb

Traceback (most recent call last):
  File "/tmp/ipykernel_62864/51583200.py", line 146, in _scrape_facebook
    html = await fb_tab.get_content()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/duoc/workflow/api/.venv/lib/python3.12/site-packages/nodriver/core/tab.py", line 1114, in get_content
    doc: cdp.dom.Node = await self.send(cdp.dom.get_document(-1, True))
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/duoc/workflow/api/.venv/lib/python3.12/site-packages/nodriver/core/connection.py", line 428, in send
    await ws.send(json.dumps(message))
  File "/home/duoc/workflow/api/.venv/lib/python3.12/site-packages/websockets/asyncio/connection.py", line 478, in send
    async with self.send_context():
  File "/usr/lib/python3.12/contextlib.py", line 210, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/duoc/workflow/api/.venv/lib/python3.12/site-packages/websockets/asyncio/connection.py", line 965, 